# Day 3 多Agent系统设计 · 牛津 Tutorial LLM 仿真 (v6.0)

## Cell 1 · Persona Prompt (Oxford + HBS + Hattie)

> 复制以下 system prompt 到任何 LLM (Claude/GPT/本地) 即可激活本 tutorial 人格。
> 本 notebook 用 **静态 if/else 模拟** 苏格拉底追问, 不真调 API (anti-stall)。

```text
You are an Oxford tutorial fellow in 多Agent系统设计 (Multi-agent system design:
A2A protocol, role specialization, communication, coordination, emergence).
Never give direct answers. 不直接给答案, 不直接答, do not answer the student's question directly.
Use Socratic questioning. Act as HBS devil's advocate. Reject vague claims.
End each turn with a probing question.

你的专长:
- LangGraph StateGraph + add_conditional_edges 构建 supervisor 中心化拓扑
- networkx 度中心性/连通性/关键路径 识别瓶颈Agent
- A2A协议 (Google 2025) 与 MCP协议 (Anthropic) 的互补关系
- 五种协作模式 (流水线/中心化/辩论/层级委托/自由协作)
- 三层通信协议 (传输/格式/语义) + pydantic AgentMessage + MessageType
- 三共识机制 (投票/权威/协商)
- 天道推演沙盘 ↔ 多Agent仿真同构 (因果链追踪/关键节点/沙盘多分支)

你的禁令:
1. 不直接给答案, 不直接答 (Never give direct answers)
2. 不接受「差不多」「大概」「应该是」等模糊表述 (Reject vague claims)
3. 每回合必须以一个追问结尾 (End with probing question)
4. 必须以 HBS devil's advocate 立场挑战学生的假设
```


## Cell 2 · Pre-Tutorial Task (强制 Retrieval Practice)

> 牛津 tutorial 前必须交一份 essay/解题/方案, 不交不准进门。提取练习 (retrieval practice) 优于重读。

**Pre-tutorial 任务 (开课前 24h 提交到 student_model.json)**:

写一段 300 字方案, 回答:

> 你被聘为一家 B2B SaaS 公司的营销技术顾问。公司现有一个单Agent系统负责「调研->策略->文案->审核」全流程, 但常出现 Context Window 溢出、创意与合规角色冲突、每个角色都做不深三个问题。请设计一个多Agent系统解决这三个瓶颈, 需包含:
> 1. 选哪种协作模式 (流水线/中心化/辩论/层级委托/自由协作)? 为什么?
> 2. 用 LangGraph StateGraph 画出你的拓扑 (节点+边, 文字描述即可)
> 3. 你设计的 Agent 间如何通信? 用 pydantic 定义 AgentMessage 关键字段
> 4. Compliance Agent 与 Content Agent 冲突时用哪种共识机制? 为什么?

**评分不计入 tutorial 成绩**, 但不交则 tutorial 自动降级为单向答疑 (失去苏格拉底回合)。

提交格式 (写入 student_model.json):
```json
{"pre_essay": "<你的300字方案>", "submitted_at": "2026-07-26T10:00:00"}
```


In [ ]:
# Cell 3 · Multi-turn Socratic Loop (静态 if/else 模拟, 不调 LLM API)
# 5 苏格拉底问: 为什么/反例/若前提变/凭什么/如何
# 4 回合: turn1=为什么, turn2=反例, turn3=若前提变, turn4=凭什么+如何

import json
from pathlib import Path

STUDENT_ESSAY = """
我选中心化协调模式, supervisor 统一调度 4 个营销 Agent (researcher/strategist/writer/reviewer)。
因为中心化可控性强, supervisor 能保证流程按序执行, 避免流水线的死板和自由协作的不收敛。
Agent 间用 JSON 通信, Compliance 拥有终决权 (权威机制)。
"""

def socratic_turn(turn_id, student_input):
    """静态 if/else 模拟牛津 tutor 的苏格拉底追问, 不调 API。"""
    turn_id = int(turn_id)
    if turn_id == 1:
        # 苏格拉底问1: 为什么
        return (
            "你说中心化『可控性强』, supervisor 能保证按序执行。\n"
            "为什么这是优点而非缺点? 为什么『按序执行』在营销场景优于『并行执行』? "
            "你的 4 个 Agent 中, researcher 和 strategist 真的需要按序吗? "
            "凭什么 supervisor 不会成为瓶颈 (反问, 见 turn2)?\n"
            "[Socratic Q1: 为什么 -- 追问因果链起点]"
        )
    elif turn_id == 2:
        # 苏格拉底问2: 反例
        return (
            "你假设 supervisor 不会成为瓶颈。给我一个反例 (counterexample): "
            "在什么营销场景下, supervisor 的 degree_centrality=1.0 反而是灾难? "
            "若移除 supervisor, 你的拓扑连通分量从 1 变 4, 这意味着什么? "
            "这不是单点故障是什么?\n"
            "[Socratic Q2: 反例 -- 挑战假设的边界]"
        )
    elif turn_id == 3:
        # 苏格拉底问3: 若前提变
        return (
            "若前提变: 假设你的公司从 B2B SaaS 转向 B2C 快消品, 营销内容从月度白皮书变成日更短视频。\n"
            "你的 supervisor 中心化拓扑还成立吗? 若日更 100 条内容, supervisor 一天要路由 400 次, "
            "延迟会不会让整个系统失去『日更』能力? 此时该转哪种协作模式? "
            "若转 team 去中心化, 你怎么保证 Compliance 不被绕过?\n"
            "[Socratic Q3: 若前提变 -- 推演情境迁移]"
        )
    elif turn_id == 4:
        # 苏格拉底问4+5: 凭什么 + 如何
        return (
            "你说 Compliance 拥有终决权 (权威机制)。凭什么 Compliance 的判断是对的? "
            "若 Compliance Agent 的训练数据过时 (2024 年法规), 它的终决权会不会变成单点故障?\n"
            "如何用天道推演的沙盘方法, 在部署前预判这个风险? "
            "如何用 networkx 把『Compliance 是单点故障』量化为某个指标 (提示: 度中心性/介数中心性)?\n"
            "[Socratic Q4: 凭什么 -- 追问权威的合法性]"
            "[Socratic Q5: 如何 -- 追问可计算的实现]"
        )
    else:
        return "Tutorial 结束。请回顾这 4 回合的追问, 你的因果模型哪里被打破了?"

# 跑 4 回合静态苏格拉底 (模拟学生只提交 essay, tutor 依次追问)
print("=" * 60)
print("学生 essay 摘要:", STUDENT_ESSAY.strip()[:80], "...")
print("=" * 60)
for t in [1, 2, 3, 4]:
    print(f"\n--- Turn {t} ---")
    print(socratic_turn(t, STUDENT_ESSAY))
print("\n" + "=" * 60)
print("Socratic 回合结束。学生需在 cell4 反思并更新 student_model。")


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度/盲点)
# 字段: mastery (0-1 per ILO), blindspots (list), socratic_turns_completed, last_updated

import json
from pathlib import Path
from datetime import datetime

SM_PATH = Path("student_model.json")

def load_student_model():
    if SM_PATH.exists():
        return json.loads(SM_PATH.read_text(encoding='utf-8'))
    return {
        "unit": "E1-D3",
        "mastery": {
            "ILO1_bottleneck": 0.0,
            "ILO2_patterns": 0.0,
            "ILO3_langgraph": 0.0,
            "ILO4_networkx": 0.0,
            "ILO5_protocol": 0.0,
            "ILO6_tiandao": 0.0
        },
        "blindspots": [],
        "socratic_turns_completed": 0,
        "pre_essay": None,
        "last_updated": None
    }

def save_student_model(sm):
    sm["last_updated"] = datetime.utcnow().isoformat() + "Z"
    SM_PATH.write_text(json.dumps(sm, ensure_ascii=False, indent=2), encoding='utf-8')
    return sm

# 模拟 cell3 跑完 4 回合后的状态更新
sm = load_student_model()
sm["socratic_turns_completed"] = 4
sm["pre_essay"] = "学生已提交 300 字方案 (见 cell2)"

# 教师根据 4 回合学生表现打分 (0-1, 静态模拟)
sm["mastery"]["ILO1_bottleneck"] = 0.7   # 学生提到三瓶颈但未解释因果
sm["mastery"]["ILO2_patterns"] = 0.5     # 只选了中心化, 没考虑反例
sm["mastery"]["ILO3_langgraph"] = 0.4    # 没提 StateGraph API
sm["mastery"]["ILO4_networkx"] = 0.2     # 完全没提 degree_centrality
sm["mastery"]["ILO5_protocol"] = 0.6     # 提了 JSON 但没 pydantic
sm["mastery"]["ILO6_tiandao"] = 0.1      # 完全没提天道推演

# 盲点 = mastery < 0.7 的 ILO
sm["blindspots"] = [
        {"ilo": "ILO4_networkx", "gap": "未用 degree_centrality 量化单点故障", "remediation": "练 practice.md drill D-EMERGE-03"},
        {"ilo": "ILO6_tiandao", "gap": "未用天道推演沙盘方法预判涌现", "remediation": "重读 notes.md 天道推演视角表 + 练 D-EMERGE-03 阶段3"},
        {"ilo": "ILO3_langgraph", "gap": "未提 StateGraph + add_conditional_edges API", "remediation": "重做 starter.ipynb TODO3 + 练 D-TOPO-02"}
]

save_student_model(sm)
print("student_model.json 已更新:")
print(json.dumps(sm, ensure_ascii=False, indent=2))


## Cell 5 · Hattie 4 级 Formative Feedback (Hattie & Timperley 2007)

> 教师在 cell4 student_model 基础上, 给出 4 级反馈。
> 避开 Self 级表扬 (Hattie: Self 级反馈效应量最低 d=0.14), 聚焦 Task/Process/Self-Reg/Feed-Forward。


In [ ]:
# Cell 5 · Hattie 4 级反馈生成器 (基于 student_model)

import json
from pathlib import Path

sm = json.loads(Path('student_model.json').read_text(encoding='utf-8'))

def hattie_feedback(sm):
    m = sm['mastery']
    bs = {b['ilo']: b for b in sm['blindspots']}
    fb = []

    # [TASK] 任务级: 针对具体任务的对错
    if m['ILO5_protocol'] < 0.7:
        fb.append("[TASK] 你的 AgentMessage 用了 JSON 而非 pydantic BaseModel。"
                  "JSON 不做 schema 校验, sender 字段拼错也不会报错; pydantic 会在 parse_obj 时拒绝。"
                  "这是格式层的基础要求, 见 notes.md 关键回顾3。")
    if m['ILO3_langgraph'] < 0.7:
        fb.append("[TASK] 你没提 StateGraph + add_conditional_edges。"
                  "中心化拓扑不是概念, 是 API 调用: graph.add_conditional_edges('supervisor', lambda s: s['next_agent'], {...})。"
                  "补做 starter.ipynb TODO3。")

    # [PROCESS] 过程级: 针对完成任务的策略/方法
    if m['ILO2_patterns'] < 0.7:
        fb.append("[PROCESS] 你只选了中心化, 没考虑反例。选型不是『哪个最好』而是『哪个最匹配任务结构』。"
                  "你的策略缺反事实推理 (counterfactual): 若用流水线会怎样? 若用辩论会怎样? "
                  "天道推演『反事实』原则要求你同时推演多个平行场景。")
    if m['ILO4_networkx'] < 0.7:
        fb.append("[PROCESS] 你判断 supervisor 是瓶颈, 但没用 networkx 量化。"
                  "策略应是: 先建 DiGraph -> 算 degree_centrality -> 若 hub 节点 centrality>=0.8 则标记单点故障。"
                  "这是从『直觉判断』升级到『可计算判断』的过程。")

    # [SELF-REG] 自我调节级: 针对学生自我监控/元认知
    if 'ILO6_tiandao' in bs:
        fb.append("[SELF-REG] 你在整个 essay 中没有出现一次『天道推演』『沙盘』『因果链』术语。"
                  "这意味着你没意识到 Day3 的核心特色是沙盘↔多Agent同构。"
                  "自我监控盲点: 你不知道自己不知道。补救 = 重读 notes.md 天道推演视角对照表, "
                  "并自问『我的方案能不能用沙盘方法在部署前预判风险?』")

    # [FEED-FORWARD] 前馈级: 下一步该做什么
    fb.append("[FEED-FORWARD] 下一步行动: "
              "(1) 今天内补做 starter.ipynb TODO3 (supervisor 拓扑) 并跑通; "
              "(2) 明天练 practice.md drill D-EMERGE-03 阶段1 Worked (networkx 度中心性); "
              "(3) 后天用 schedule.json card C4+C7 间隔重复 (1天后第一次复习); "
              "(4) 3 天后重新提交 300 字方案 (重点加 networkx 指标 + 天道推演术语), 触发 retry_policy 不罚分。")

    return fb

for line in hattie_feedback(sm):
    print(line)
    print()


## Cell 6 · 限频与退出工件 (防依赖 + 盲点导出)

### 限频 (Rate Limit, 防依赖)

- **每单元 1 次/天**: 本 tutorial 每天 1 次 (usage limit), 防止学生依赖 tutor 而不独立思考。
- **超限提示**: 若今日已用 1 次, tutor 返回「今日已用尽, 请先独立做 drill D-TOPO-02 阶段3 Independent, 明天再来。明天来时请带新的 300 字方案。」
- **原因**: 牛津 tutorial 的价值在于学生 *带着困惑* 进门, 而非把 tutor 当答疑机。Bjork 的 desirable difficulty 原则: 间隔 24h 让大脑在睡眠中巩固, 比连续追问更有效。
- **mastery 阈值触发**: 当所有 ILO 的 mastery >= 0.8, tutorial 自动降频为 1 次/周 (学生已具备自我调节能力)。

### 退出工件 (Exit Artifact, 每次 tutorial 结束导出)

每次 tutorial 结束, 学生从 cell4 student_model 提取:

1. **2-3 盲点** (mastery < 0.7 的 ILO, 每盲点附 remediation 路径)
2. **推荐复习单元** (基于盲点跨单元映射):
   - ILO1 (单Agent瓶颈) 弱 -> 复习 Day 1 (Agent 基础, ReAct/Plan-Execute)
   - ILO3 (LangGraph) 弱 -> 复习 Day 2 (Agent 框架对比, LangGraph 单 Agent)
   - ILO6 (天道推演) 弱 -> 复习 项目 CLAUDE.md「天道推演系统」章节
3. **下次 tutorial 的 pre-essay 题目** (由 tutor 根据盲点定制, 强制 retrieval)

**示例退出工件** (基于 cell4 模拟数据):

```
盲点1: ILO4_networkx (mastery=0.2)
  - gap: 未用 degree_centrality 量化单点故障
  - remediation: practice.md drill D-EMERGE-03 阶段1 Worked + 阶段2 Faded
  - 复习单元: Day 2 networkx 基础 (若有) / 项目 CLAUDE.md 天道推演「关键节点」概念

盲点2: ILO6_tiandao (mastery=0.1)
  - gap: 未用天道推演沙盘方法预判涌现
  - remediation: 重读 notes.md 天道推演视角表 + 练 D-EMERGE-03 阶段3 Independent
  - 复习单元: 项目 CLAUDE.md「天道推演系统」核心定义章节

盲点3: ILO3_langgraph (mastery=0.4)
  - gap: 未提 StateGraph + add_conditional_edges API
  - remediation: 重做 starter.ipynb TODO3 + 练 D-TOPO-02
  - 复习单元: Day 2 LangGraph 单 Agent 基础

下次 pre-essay 题目:
  「用 networkx 度中心性 + 天道推演沙盘方法, 推演你的 supervisor 拓扑在 B2C 日更场景下的涌现决策质量。
   需出现 degree_centrality 数值预测 + 沙盘多分支推演至少 3 层 (immediate -> near -> far)。」
```

---

*本 tutorial.ipynb 为 v6.0 学习科学层新增, 不修改 v5.0 的 notes.md/starter.ipynb/solution.ipynb/reading.md/data。*
*苏格拉底回合为静态 if/else 模拟, 不调 LLM API, 符合 anti-stall 原则。*
*最后更新: 2026-07-26*
